In [1]:
from __future__ import annotations

import os

from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from torch_geometric.data import Data, Dataset as PyGDataset

from qqe.GNN.physics_aware_NN import GNN
import torch.nn as nn
from torch.optim import Adam
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader

In [2]:
class QuantumCircuitGraphDataset(PyGDataset):
    """Loads per-circuit .pt files produced by compute_entry_for_config.

    Each sample file is expected to contain:
      - x: Tensor [N, node_dim]
      - edge_index: Tensor [2, E]
      - global_features: Tensor [G]
      - sre: float (label)
      - gate_counts: dict (optional, carried along)
      - meta: dict (optional)
    """

    def __init__(
        self,
        root: str,
        pt_paths: list[str],
        global_feature_variant: str = "baseline",
        node_feature_backend_variant: str | None = None,
        transform=None,
        pre_transform=None,
    ):
        self.pt_paths = [str(p) for p in pt_paths]
        self.global_feature_variant = global_feature_variant
        self.node_feature_backend_variant = node_feature_backend_variant
        super().__init__(root=root, transform=transform, pre_transform=pre_transform)

    @property
    def raw_file_names(self) -> list[str]:
        # Not used (we bypass PyG raw/processed system), but required by base class.
        return []

    @property
    def processed_file_names(self) -> list[str]:
        return []
    def len(self) -> int:
        return len(self.pt_paths)

    def get(self, idx: int) -> Data:
        obj = torch.load(self.pt_paths[idx], map_location="cpu")

        x = obj["x"]
        edge_index = obj["edge_index"]
        g = obj["global_features"]
        y_val = obj.get("sre", None)

        # Dtypes for model
        if x.dtype != torch.float32:
            x = x.to(torch.float32)
        if edge_index.dtype != torch.long:
            edge_index = edge_index.to(torch.long)
        if g.dtype != torch.float32:
            g = g.to(torch.float32)

        # label
        if y_val is None:
            y = torch.tensor([float("nan")], dtype=torch.float32)
        else:
            y = torch.tensor([float(y_val)], dtype=torch.float32)

        data = Data(x=x, edge_index=edge_index, y=y)
        data.global_features = g
        data.num_qubits = int(obj.get("meta", {}).get("n_qubits", 0))

        # Keep for debugging / analysis (won't hurt)
        data.gate_counts = obj.get("gate_counts", {})
        data.meta = obj.get("meta", {})

        return data

In [3]:
def collect_pt_paths(dataset_dir: str) -> list[str]:
    d = Path(dataset_dir)
    # support either dataset_dir/*.pt or dataset_dir/samples/*.pt
    paths = sorted((d / "encoding_data_pennylane_fwht").glob("*.pt"))
    if not paths:
        paths = sorted(d.glob("*.pt"))
    return [str(p) for p in paths]

In [4]:
import hashlib


def _cache_root_for_paths(paths: list[str], suffix: str = "") -> str:
    canonical = "|".join(sorted(Path(p).name for p in paths))

    digest = hashlib.md5(canonical.encode("utf-8")).hexdigest()[:10]

    tag = f"_{suffix}" if suffix else ""

    cache_dir = Path("qqe") / "cache" / f"pyg_cache_{digest}{tag}"
    cache_dir.mkdir(parents=True, exist_ok=True)

    return str(cache_dir)

In [5]:
def get_node_feature_dim_from_sample(pt_paths: list[str]) -> int:
    obj = torch.load(pt_paths[0], map_location="cpu")
    return int(obj["x"].shape[1])

def get_global_feature_dim_from_sample(pt_paths: list[str]) -> int:
    obj = torch.load(pt_paths[0], map_location="cpu")
    return int(obj["global_features"].numel())

In [6]:
def _amp_device_type() -> str:
    return "cuda" if torch.cuda.is_available() else "cpu"

In [7]:
device_type = _amp_device_type()
print(f"AMP device type: {device_type}")

AMP device type: cpu


In [8]:
def _safe_y(batch) -> torch.Tensor:
    """Return y as float tensor shaped [num_graphs]."""
    if not hasattr(batch, "y") or batch.y is None:
        raise RuntimeError("Batch is missing labels 'y'. Make sure your dataset sets data.y.")
    y = batch.y
    if y.dim() == 0:
        y = y.view(1)
    if y.dim() == 2 and y.size(1) == 1:
        y = y.view(-1)
    return y.float()

In [9]:
@torch.no_grad()
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    loss_fn: nn.Module,
    use_amp: bool = True,
) -> float:
    model.eval()
    total_loss = 0.0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device, non_blocking=True)
        y = _safe_y(batch)

        with autocast(
            device_type=device_type,
            enabled=(use_amp and device.type == "cuda"),
        ):
            pred = model(batch).view(-1).float()

            # (Optional) ignore NaN labels if you ever have any
            mask = torch.isfinite(y)
            if mask.sum() == 0:
                continue
            loss = loss_fn(pred[mask], y[mask])

        total_loss += float(loss.item()) * int(batch.num_graphs)
        total_graphs += int(batch.num_graphs)

    return total_loss / max(1, total_graphs)


@dataclass
class TrainHistory:
    train_loss: list[float]
    val_loss: list[float]
    lr: list[float]


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    *,
    epochs: int = 200,
    lr: float = 1e-3,
    weight_decay: float = 0.0,
    device: str | None = None,
    loss_type: str = "huber",  # "mse" | "huber"
    huber_delta: float = 1.0,
    grad_clip: float = 5.0,
    early_stopping_patience: int = 15,
    early_stopping_min_delta: float = 0.0,
    use_amp: bool = True,
    scheduler: str = "none",   # "none" | "plateau"
) -> tuple[nn.Module, TrainHistory, torch.device]:
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    model = model.to(dev)

    if loss_type.lower() == "mse":
        loss_fn: nn.Module = nn.MSELoss()
    elif loss_type.lower() == "huber":
        loss_fn = nn.SmoothL1Loss(beta=huber_delta)
    else:
        raise ValueError("loss_type must be 'mse' or 'huber'")

    opt = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if scheduler == "plateau":
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=5)
    else:
        sch = None

    scaler = GradScaler(
        enabled=(use_amp and dev.type == "cuda"),
    )

    hist = TrainHistory(train_loss=[], val_loss=[], lr=[])

    best_val = float("inf")
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        print(f"-------- EPOCH : {epoch:03d} --------\n")
        model.train()
        total_loss = 0.0
        total_graphs = 0

        for batch in train_loader:
            batch = batch.to(dev, non_blocking=True)
            y = _safe_y(batch)

            opt.zero_grad(set_to_none=True)

            with autocast(device_type=device_type, enabled=(use_amp and dev.type == "cuda")):
                pred = model(batch).view(-1).float()
                mask = torch.isfinite(y)
                if mask.sum() == 0:
                    continue
                loss = loss_fn(pred[mask], y[mask])

            if not torch.isfinite(loss):
                continue

            scaler.scale(loss).backward()

            if grad_clip is not None and grad_clip > 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float(grad_clip))

            scaler.step(opt)
            scaler.update()

            total_loss += float(loss.item()) * int(batch.num_graphs)
            total_graphs += int(batch.num_graphs)

        train_loss = total_loss / max(1, total_graphs)
        val_loss = evaluate_loss(model, val_loader, dev, loss_fn, use_amp=use_amp)

        if sch is not None:
            sch.step(val_loss)

        current_lr = float(opt.param_groups[0]["lr"])
        hist.train_loss.append(float(train_loss))
        hist.val_loss.append(float(val_loss))
        hist.lr.append(current_lr)

        print(f"Losses | train {train_loss:.6f} | val {val_loss:.6f} | lr {current_lr:.2e}")

        # Early stopping
        if val_loss + early_stopping_min_delta < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= early_stopping_patience:
                print(f"Early stopping at epoch {epoch:03d} (best val {best_val:.6f})")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, hist, dev

In [10]:
def plot_training_curves(hist: TrainHistory, title: str = "Training curves"):
    epochs = list(range(1, len(hist.train_loss) + 1))

    plt.figure()
    plt.plot(epochs, hist.train_loss, label="train")
    plt.plot(epochs, hist.val_loss, label="val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

    if hist.lr:
        plt.figure()
        plt.plot(epochs, hist.lr, label="lr")
        plt.xlabel("Epoch")
        plt.ylabel("Learning rate")
        plt.title("Learning rate")
        plt.grid(True)
        plt.show()

In [11]:
def build_train_test_loaders(
    pt_paths: list[str],
    train_split: float = 0.8,
    batch_size: int = 64,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
) -> tuple[DataLoader, DataLoader]:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )
    if len(dataset) < 2:
        raise RuntimeError("Dataset too small to split. Check PT paths.")

    train_len = max(1, int(len(dataset) * train_split))
    test_len = max(1, len(dataset) - train_len)
    while train_len + test_len > len(dataset):
        train_len -= 1

    generator = torch.Generator().manual_seed(seed)
    train_ds, test_ds = random_split(dataset, [train_len, test_len], generator=generator)

    num_cpus = os.cpu_count() or 0
    default_workers = 2 if num_cpus > 2 else 0
    pin_mem = torch.cuda.is_available()

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=default_workers, pin_memory=pin_mem),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=default_workers, pin_memory=pin_mem),
    )


def build_full_loader(
    pt_paths: list[str],
    batch_size: int = 64,
    global_feature_variant: str = "binned",
    node_feature_backend_variant: str | None = None,
) -> DataLoader:
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )
    if len(dataset) == 0:
        raise RuntimeError("Dataset is empty. Check PT paths and formats.")

    num_cpus = os.cpu_count() or 0
    default_workers = 2 if num_cpus > 2 else 0
    pin_mem = torch.cuda.is_available()

    return DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=default_workers, pin_memory=pin_mem)


def build_train_val_test_loaders_two_stage(
    pt_paths: list[str],
    train_split: float = 0.8,
    val_within_train: float = 0.1,
    batch_size: int = 32,
    seed: int = 42,
    global_feature_variant: str = "baseline",
    node_feature_backend_variant: str | None = None,
):
    suffix = f"{global_feature_variant}_backend_{node_feature_backend_variant or 'none'}"
    root = _cache_root_for_paths(pt_paths, suffix=suffix)

    dataset = QuantumCircuitGraphDataset(
        root=root,
        pt_paths=pt_paths,
        global_feature_variant=global_feature_variant,
        node_feature_backend_variant=node_feature_backend_variant,
    )
    if len(dataset) < 3:
        raise RuntimeError("Dataset too small for train/val/test splitting.")

    generator = torch.Generator().manual_seed(seed)
    primary_train_len = max(1, int(len(dataset) * train_split))
    test_len = max(1, len(dataset) - primary_train_len)
    while primary_train_len + test_len > len(dataset):
        primary_train_len -= 1

    primary_train, test_ds = random_split(dataset, [primary_train_len, test_len], generator=generator)

    val_len = max(1, int(len(primary_train) * val_within_train))
    real_train_len = max(1, len(primary_train) - val_len)
    train_ds, val_ds = random_split(primary_train, [real_train_len, val_len], generator=generator)

    num_cpus = os.cpu_count() or 0
    default_workers = 2 if num_cpus > 2 else 0
    pin_mem = torch.cuda.is_available()

    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=default_workers, pin_memory=pin_mem),
        DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=default_workers, pin_memory=pin_mem),
        DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=default_workers, pin_memory=pin_mem),
    )

In [12]:
pt_paths = collect_pt_paths("..\\qqe\\data")  # or .../samples
train_loader, val_loader, test_loader = build_train_val_test_loaders_two_stage(
    pt_paths,
    global_feature_variant="binned",
    batch_size=32,
)

node_in_dim = get_node_feature_dim_from_sample(pt_paths)
global_in_dim = get_global_feature_dim_from_sample(pt_paths)

In [13]:
print(val_loader.dataset[0])

Data(
  x=[441, 23],
  edge_index=[2, 542],
  y=[1],
  global_features=[153],
  num_qubits=10,
  gate_counts={
    rx_bin_0=1,
    rx_bin_1=3,
    rx_bin_2=0,
    rx_bin_3=3,
    rx_bin_4=3,
    rx_bin_5=2,
    rx_bin_6=2,
    rx_bin_7=3,
    rx_bin_8=2,
    rx_bin_9=3,
    rx_bin_10=1,
    rx_bin_11=7,
    rx_bin_12=0,
    rx_bin_13=3,
    rx_bin_14=3,
    rx_bin_15=2,
    rx_bin_16=4,
    rx_bin_17=1,
    rx_bin_18=1,
    rx_bin_19=0,
    rx_bin_20=3,
    rx_bin_21=4,
    rx_bin_22=3,
    rx_bin_23=1,
    rx_bin_24=2,
    rx_bin_25=3,
    rx_bin_26=0,
    rx_bin_27=2,
    rx_bin_28=3,
    rx_bin_29=1,
    rx_bin_30=1,
    rx_bin_31=1,
    rx_bin_32=2,
    rx_bin_33=1,
    rx_bin_34=1,
    rx_bin_35=7,
    rx_bin_36=1,
    rx_bin_37=2,
    rx_bin_38=1,
    rx_bin_39=2,
    rx_bin_40=3,
    rx_bin_41=3,
    rx_bin_42=1,
    rx_bin_43=1,
    rx_bin_44=2,
    rx_bin_45=3,
    rx_bin_46=2,
    rx_bin_47=2,
    rx_bin_48=2,
    rx_bin_49=0,
    ry_bin_0=4,
    ry_bin_1=2,
    ry_bin_2=2,
 

In [14]:
model = GNN(
    node_in_dim=node_in_dim,
    global_in_dim=global_in_dim,
    gnn_hidden=32,
    gnn_heads=8,
    global_hidden=16,
    reg_hidden=16,
    num_layers=5,
    dropout_rate=0.1,
)

In [15]:
EPOCHS = 10
l_r = 0.001
loss = "mse"

In [ ]:
model, hist, dev = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    lr=l_r,
    loss_type=loss,
    huber_delta=1.0,
    early_stopping_patience=15,
    scheduler="plateau",
)

-------- EPOCH : 001 --------



C:\Users\victo\AppData\Local\Temp\ipykernel_3940\389923350.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(


In [ ]:
plot_training_curves(hist, title="GNN SRE regression")